# Woodpile beam-spreading — retrieval

Downloads the field data of `20260918_Beam_Spreading_Experiment_woodpiles` (structure run + empty-box reference) and
saves a compact `.h5` under `./data`. Mirrors `../Transmission/20260918_Transmission_retrieval.ipynb` and the LSU
`20260601_Beam_Spreading_freq/20251007_Retireve_Field_Data.ipynb`.

Per structure (key = task stem) the file holds:

* `x`, `y` — exit-monitor coordinates (µm), `freqs` (Hz), `lambda` (µm), `nu = d/lambda`;
* `I_exit[ν, x, y] = |Ex|²+|Ey|²+|Ez|²` (float32) of the structure, `I_exit_ref` (empty box), `I_entry` (entrance
  monitor, structure run) and `I_entry_ref`;
* the complex exit-face fields `Ex, Ey, Ez` (complex64) of the structure run and `Ex_ref, Ey_ref, Ez_ref` of the empty
  box — needed for the time-domain reconstruction — kept on the full monitor grid while the file stays below ~2 GB,
  otherwise decimated `::2` in x and y (recorded in `complex_decimation`);
* `params` and the `defects` table copied from the structure `.h5` (never `epsilon`).

The notebook is incremental: keys already present in the output file are skipped, so it can be re-run as tasks finish.


In [ ]:
import os, sys, re
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
import numpy as np
import tidy3d as td
from tidy3d import web
from natsort import natsorted

sys.path.append(os.path.abspath(r'H:\codes\tidy3d'))
import AutomationModule as AM
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [ ]:
project_name = "20260918_Beam_Spreading_Experiment_woodpiles"
folder_path = rf"{Path.cwd().parents[2]}/data/{project_name}"       # task-id .txt files written by the experiment notebook
structures_path = Path("../Structures")                               # *_tables.h5 (params, defects)
out_file = Path("./data/20260918_beam_spreading_data_woodpiles.h5")
size_budget_bytes = 2e9          # keep complex fields on the full grid while the file stays below this
os.makedirs(out_file.parent, exist_ok=True)

data = AM.read_hdf5_as_dict(str(out_file)) if out_file.exists() else {}
print("already stored:", list(data.keys()))


In [ ]:
def intensity(fd):
    return (np.abs(fd.Ex.values)**2 + np.abs(fd.Ey.values)**2 + np.abs(fd.Ez.values)**2).astype(np.float32)

def structure_file_for(stem):
    '''Match a task stem '<structure stem>_lambda_<lmin>_<lmax>' back to its *_tables.h5.'''
    base = re.sub(r"_lambda_.*$", "", stem)
    hits = [p for p in structures_path.rglob("*_tables.h5") if p.stem == base]
    if len(hits) != 1:
        raise FileNotFoundError(f"{len(hits)} structure files match stem {base!r}: {hits}")
    return hits[0]

def nbytes_entry(entry):
    return sum(v.nbytes for v in entry.values() if isinstance(v, np.ndarray))

total_bytes = sum(nbytes_entry({k: v for k, v in e.items() if isinstance(v, np.ndarray)})
                  for e in data.values() if isinstance(e, dict))

for dirpath, dirnames, filenames in os.walk(folder_path):
    for filename in natsorted(filenames):
        file = os.path.join(dirpath, filename)
        if not (os.path.isfile(file) and filename.endswith(".txt")):
            continue
        key = Path(filename).stem
        print(filename)
        if key in data:
            print("  already stored, skipping")
            continue

        structure_1 = AM.loadFromFile(key=tidy3dAPI, file_path=file, get_ref=True)
        sim_data, sim_data0 = structure_1.sim_data, structure_1.sim_data0
        fd_exit = sim_data.monitor_data["monitorField_exit"]
        fd_entry = sim_data.monitor_data["monitorField_entry"]
        fd_exit0 = sim_data0.monitor_data["monitorField_exit"]
        fd_entry0 = sim_data0.monitor_data["monitorField_entry"]

        sfile = structure_file_for(key)
        sdata = AM.read_hdf5_as_dict(str(sfile))
        params, defects = sdata["params"], sdata["defects"]
        d = float(params["d"])
        freqs = fd_exit.Ex.f.values
        lam = td.C_0 / freqs

        # field arrays are (x, y, z=1, f): squeeze z and move f first -> (f, x, y)
        def fxy(arr):
            return np.moveaxis(np.squeeze(arr, axis=2), -1, 0)

        entry = {
            "x": fd_exit.Ex.x.values, "y": fd_exit.Ex.y.values, "z_exit": fd_exit.Ex.z.values,
            "x_entry": fd_entry.Ex.x.values, "y_entry": fd_entry.Ex.y.values, "z_entry": fd_entry.Ex.z.values,
            "freqs": freqs, "lambda": lam, "nu": d / lam,
            "I_exit": fxy(intensity(fd_exit)), "I_exit_ref": fxy(intensity(fd_exit0)),
            "I_entry": fxy(intensity(fd_entry)), "I_entry_ref": fxy(intensity(fd_entry0)),
            "run_time": np.float64(sim_data.simulation.run_time),
            "cost_flexcredits": np.asarray(structure_1.cost if isinstance(structure_1.cost, (int, float)) else np.nan),
            "task_ids": np.asarray(structure_1.list_id, dtype=str),
            "structure_file": np.asarray(str(sfile.name)),
            "params": {k: np.asarray(v) for k, v in params.items()},
            "defects": {k: np.asarray(v) for k, v in defects.items()},
        }
        # complex exit fields (structure + reference), full grid if the budget allows, else ::2
        cplx_full = {}
        for lab, fd in (("", fd_exit), ("_ref", fd_exit0)):
            for comp in ("Ex", "Ey", "Ez"):
                cplx_full[comp + lab] = fxy(getattr(fd, comp).values.astype(np.complex64))
        if total_bytes + nbytes_entry(entry) + sum(v.nbytes for v in cplx_full.values()) <= size_budget_bytes:
            entry.update(cplx_full); entry["complex_decimation"] = np.int64(1)
        else:
            entry.update({k: v[:, ::2, ::2] for k, v in cplx_full.items()}); entry["complex_decimation"] = np.int64(2)
            entry["x_complex"] = entry["x"][::2]; entry["y_complex"] = entry["y"][::2]
            print("  size budget exceeded -> complex fields stored on the ::2 grid")
        total_bytes += nbytes_entry(entry) + sum(v.nbytes for v in cplx_full.values()) / entry["complex_decimation"]**2
        data[key] = entry
        print(f"  stored {key}: {fd_exit.Ex.shape} field, {len(freqs)} freqs, "
              f"nu in [{entry['nu'].min():.3f}, {entry['nu'].max():.3f}], decimation {entry['complex_decimation']}, "
              f"cumulative ~{total_bytes/1e9:.2f} GB")


In [ ]:
AM.create_hdf5_from_dict(data, str(out_file))
print("saved", out_file, f"{out_file.stat().st_size/1e9:.2f} GB", "keys:", list(data.keys()))
